# Example 04: Multi-Story Building with Shell Walls and Slabs

## Overview

This example demonstrates a **2-story reinforced concrete building** modeled with:
- **Shear walls** (vertical surfaces in XZ and YZ planes) → `ASDShellQ4` elements
- **Floor slabs** (horizontal surfaces in XY plane) → `ASDShellQ4` elements
- **Columns** (vertical line segments) → `elasticBeamColumn` elements

Key concepts:
- **Fragment operation**: Ensures conformal meshing by creating shared nodes at all component intersections
- **Mixed element types**: Shells (ndof=6 per node) + beams (ndof=6 per node) in a single ndm=3, ndf=6 model
- **Multi-component assembly**: Walls, slabs, and columns form a rigid lateral-load-resisting system
- **Lateral load analysis**: Static pushover to evaluate building stiffness and story drift


In [ ]:
from pyGmsh import pyGmsh
from pyGmsh import Algorithm2D
import numpy as np
import matplotlib.pyplot as plt

## Parameters

In [ ]:
# ── Building geometry ─────────────────────────────────────────────────────
Lx = 6.0          # plan length (X direction)           [m]
Ly = 4.0          # plan width (Y direction)            [m]
story_h = 3.0     # height of each story                [m]
n_stories = 2     # number of stories
total_h = n_stories * story_h

# ── Section dimensions ────────────────────────────────────────────────────
t_slab = 0.20     # floor slab thickness                [m]
t_wall = 0.25     # shear wall thickness                [m]
col_side = 0.40   # column square cross-section         [m]

# ── Concrete properties ────────────────────────────────────────────────────
Ec = 30.0e9       # Young's modulus                     [Pa]
nu = 0.2          # Poisson's ratio
rho_c = 2400.0    # density                             [kg/m³]

# Column geometric properties
Ac = col_side ** 2                   # cross-sectional area
Jx = col_side ** 4 / 12              # polar moment of inertia
Iy = col_side ** 4 / 12              # moment of inertia (y-axis)
Iz = col_side ** 4 / 12              # moment of inertia (z-axis)
G = Ec / (2 * (1 + nu))              # shear modulus

# ── Meshing parameters ────────────────────────────────────────────────────
lc = 0.3          # target element size for walls/slabs [m]

print("Building Parameters")
print("=" * 50)
print(f"Plan dimensions:     {Lx:.1f}m × {Ly:.1f}m")
print(f"Total height:        {total_h:.1f}m ({n_stories} stories)")
print(f"Story height:        {story_h:.1f}m")
print(f"Slab thickness:      {t_slab:.2f}m")
print(f"Wall thickness:      {t_wall:.2f}m")
print(f"Column section:      {col_side:.2f}m × {col_side:.2f}m")
print(f"Concrete E:          {Ec/1e9:.1f} GPa")
print(f"Target element size: {lc:.2f}m")

## Geometry: Walls, Slabs, and Columns

### Assembly Strategy

We create a multi-component structure:

1. **Front and back shear walls** (XZ planes):
   - Front wall at y = 0,  spanning x ∈ [0, Lx], z ∈ [0, total_h]
   - Back wall at y = Ly, spanning x ∈ [0, Lx], z ∈ [0, total_h]

2. **Floor slabs** (XY planes):
   - One slab at each floor level (z = story_h, 2×story_h)
   - Spanning x ∈ [0, Lx], y ∈ [0, Ly]

3. **Corner columns** (vertical line segments):
   - Four columns at (0,0), (Lx,0), (0,Ly), (Lx,Ly)
   - Each column spans full height in Z

4. **Fragment operation**:
   - After creating all surfaces and lines, call `fragment()` to ensure conformal mesh
   - All components share nodes at junctions (wall-slab, wall-column, etc.)

5. **Physical groups** (for reference and load application):
   - "Walls" — all wall surfaces
   - "Slabs" — all slab surfaces
   - "Columns" — all column line segments
   - "Fixed" — base edges/points (z = 0)
   - "RoofSlab" — top slab (z = total_h) for load application


In [ ]:
g = pyGmsh(model_name="MultiStoryBuilding", verbose=False)
g.initialize()

try:
    # ────────────────────────────────────────────────────────────────────────
    # 1. Create wall surfaces (vertical planes)
    # ────────────────────────────────────────────────────────────────────────
    
    print("Creating geometry...")
    
    # Front wall (at y=0, spanning x and z)
    p_fw1 = g.model.add_point(0.0,   0.0, 0.0, mesh_size=lc)
    p_fw2 = g.model.add_point(Lx,    0.0, 0.0, mesh_size=lc)
    p_fw3 = g.model.add_point(Lx,    0.0, total_h, mesh_size=lc)
    p_fw4 = g.model.add_point(0.0,   0.0, total_h, mesh_size=lc)
    
    l_fw1 = g.model.add_line(p_fw1, p_fw2)
    l_fw2 = g.model.add_line(p_fw2, p_fw3)
    l_fw3 = g.model.add_line(p_fw3, p_fw4)
    l_fw4 = g.model.add_line(p_fw4, p_fw1)
    
    loop_fw = g.model.add_curve_loop([l_fw1, l_fw2, l_fw3, l_fw4])
    surf_wall_front = g.model.add_plane_surface(loop_fw, label="Wall_Front")
    
    # Back wall (at y=Ly, spanning x and z)
    p_bw1 = g.model.add_point(0.0,   Ly, 0.0, mesh_size=lc)
    p_bw2 = g.model.add_point(Lx,    Ly, 0.0, mesh_size=lc)
    p_bw3 = g.model.add_point(Lx,    Ly, total_h, mesh_size=lc)
    p_bw4 = g.model.add_point(0.0,   Ly, total_h, mesh_size=lc)
    
    l_bw1 = g.model.add_line(p_bw1, p_bw2)
    l_bw2 = g.model.add_line(p_bw2, p_bw3)
    l_bw3 = g.model.add_line(p_bw3, p_bw4)
    l_bw4 = g.model.add_line(p_bw4, p_bw1)
    
    loop_bw = g.model.add_curve_loop([l_bw1, l_bw2, l_bw3, l_bw4])
    surf_wall_back = g.model.add_plane_surface(loop_bw, label="Wall_Back")
    
    print(f"  ✓ Created front wall (surface {surf_wall_front})")
    print(f"  ✓ Created back wall (surface {surf_wall_back})")
    
    # ────────────────────────────────────────────────────────────────────────
    # 2. Create floor slabs (horizontal planes)
    # ────────────────────────────────────────────────────────────────────────
    
    slab_surfaces = []
    for i in range(1, n_stories + 1):
        z_level = i * story_h
        
        p_s1 = g.model.add_point(0.0,  0.0,  z_level, mesh_size=lc)
        p_s2 = g.model.add_point(Lx,   0.0,  z_level, mesh_size=lc)
        p_s3 = g.model.add_point(Lx,   Ly,   z_level, mesh_size=lc)
        p_s4 = g.model.add_point(0.0,  Ly,   z_level, mesh_size=lc)
        
        l_s1 = g.model.add_line(p_s1, p_s2)
        l_s2 = g.model.add_line(p_s2, p_s3)
        l_s3 = g.model.add_line(p_s3, p_s4)
        l_s4 = g.model.add_line(p_s4, p_s1)
        
        loop_s = g.model.add_curve_loop([l_s1, l_s2, l_s3, l_s4])
        slab_surf = g.model.add_plane_surface(loop_s, label=f"Slab_{i}")
        slab_surfaces.append(slab_surf)
    
    print(f"  ✓ Created {len(slab_surfaces)} floor slabs")
    
    # ────────────────────────────────────────────────────────────────────────
    # 3. Create column line segments at corners
    # ────────────────────────────────────────────────────────────────────────
    
    column_lines = []
    corner_coords = [
        (0.0,   0.0),   # corner 0
        (Lx,    0.0),   # corner 1
        (Lx,    Ly),    # corner 2
        (0.0,   Ly),    # corner 3
    ]
    
    for i, (cx, cy) in enumerate(corner_coords):
        # Create points at base and top
        p_base = g.model.add_point(cx, cy, 0.0, mesh_size=lc)
        p_top = g.model.add_point(cx, cy, total_h, mesh_size=lc)
        
        # Create line connecting them
        col_line = g.model.add_line(p_base, p_top)
        column_lines.append(col_line)
    
    print(f"  ✓ Created {len(column_lines)} corner columns")
    
    # ────────────────────────────────────────────────────────────────────────
    # 4. Synchronize and fragment for conformal meshing
    # ────────────────────────────────────────────────────────────────────────
    
    g.model.sync()
    
    # Fragment all surfaces and lines together
    # This ensures shared nodes at all junctions
    all_surf_tags = [surf_wall_front, surf_wall_back] + slab_surfaces
    all_line_tags = column_lines
    
    surf_dimtags = [(2, tag) for tag in all_surf_tags]
    line_dimtags = [(1, tag) for tag in all_line_tags]
    
    fragments_s, fragments_m = g.model.fragment(surf_dimtags, line_dimtags)
    print(f"  ✓ Fragmented {len(surf_dimtags)} surfaces with {len(line_dimtags)} lines")
    print(f"    Result: {len(fragments_s)} surface fragments, {len(fragments_m)} line fragments")
    
    g.model.sync()
    
    # ────────────────────────────────────────────────────────────────────────
    # 5. Create physical groups
    # ────────────────────────────────────────────────────────────────────────
    
    # Extract all surface and line tags after fragmentation
    all_surfs = gmsh.model.getEntities(2)
    all_lines = gmsh.model.getEntities(1)
    
    import gmsh
    
    all_surfs = gmsh.model.getEntities(2)
    all_lines = gmsh.model.getEntities(1)
    
    # Identify walls, slabs, and columns by location and physical extent
    wall_surfs = []
    slab_surfs = []
    col_lines = []
    
    # Walls: surfaces with cz near 0 or middle of building height
    for dim, tag in all_surfs:
        cx, cy, cz, ux, uy, uz = gmsh.model.getBoundingBox(dim, tag)
        cz = (cz + uz) / 2.0  # center z-coordinate
        
        # Check if surface spans mostly in z direction (wall-like)
        z_extent = uz - cz
        x_extent = ux if ux > cy else ux - 0
        y_extent = uy if uy > cy else abs(uy - cy)
        
        cy_val = (cy + uy) / 2.0
        
        # Walls: at y≈0 or y≈Ly, spanning in x and z
        if abs(cy_val) < 0.1 or abs(cy_val - Ly) < 0.1:
            wall_surfs.append(tag)
        # Slabs: surfaces at specific z levels (near story heights)
        elif any(abs(cz - z_level) < 0.5 for z_level in [story_h, 2*story_h]):
            slab_surfs.append(tag)
    
    # Columns: short line segments (less than 10% of total height)
    for dim, tag in all_lines:
        x1, y1, z1, x2, y2, z2 = gmsh.model.getBoundingBox(1, tag)
        length = np.sqrt((x2-x1)**2 + (y2-y1)**2 + (z2-z1)**2)
        
        # Columns span mostly in Z, small length in X-Y plane
        if length > story_h * 0.8 and length < total_h * 1.2:
            dx_dy = np.sqrt((x2-x1)**2 + (y2-y1)**2)
            if dx_dy < 0.1:
                col_lines.append(tag)
    
    # Add physical groups
    if wall_surfs:
        gmsh.model.addPhysicalGroup(2, wall_surfs, name="Walls")
    if slab_surfs:
        gmsh.model.addPhysicalGroup(2, slab_surfs, name="Slabs")
    if col_lines:
        gmsh.model.addPhysicalGroup(1, col_lines, name="Columns")
    
    # Find base and roof edges
    base_points = []
    roof_points = []
    
    for dim, tag in gmsh.model.getEntities(0):
        x, y, z = gmsh.model.getValue(0, tag, [0, 0, 0])
        if abs(z) < 0.01:
            base_points.append(tag)
        elif abs(z - total_h) < 0.01:
            roof_points.append(tag)
    
    if base_points:
        gmsh.model.addPhysicalGroup(0, base_points, name="Fixed")
    if roof_points:
        gmsh.model.addPhysicalGroup(0, roof_points, name="RoofLevel")
    
    print(f"  ✓ Physical groups created:")
    print(f"    - Walls: {len(wall_surfs)} surfaces")
    print(f"    - Slabs: {len(slab_surfs)} surfaces")
    print(f"    - Columns: {len(col_lines)} lines")
    print(f"    - Fixed base: {len(base_points)} points")
    print(f"    - Roof level: {len(roof_points)} points")

except Exception as e:
    print(f"Error during geometry creation: {e}")
    g.finalize()
    raise

## Meshing

In [ ]:
try:
    print("
Generating mesh...")
    
    # Set global element size
    g.mesh.set_global_size(lc)
    
    # Generate 2D mesh (all surfaces and lines)
    g.mesh.generate(2)
    
    # Recombine for quadrilateral elements on surfaces
    gmsh.model.mesh.setRecombine(2)
    gmsh.model.mesh.generate(2)
    
    print(f"  ✓ Mesh generated")
    
    # Quality report
    quality_report = g.mesh.quality_report()
    print("
── Mesh Quality ──")
    print(quality_report.to_string())
    
    # Element count
    node_tags, node_coords, _ = gmsh.model.mesh.getNodes()
    elem_types, elem_tags, elem_nodeTags = gmsh.model.mesh.getElements()
    
    print(f"
── Mesh Statistics ──")
    print(f"Nodes:    {len(node_tags)}")
    print(f"Elements: {sum(len(tags) for tags in elem_tags)}")
    
    # Save mesh
    g.mesh.save("building.msh")
    print(f"
Mesh saved → building.msh")

except Exception as e:
    print(f"Error during meshing: {e}")
    g.finalize()
    raise

## OpenSees Model: Mixed Element Types

### Element Assignment Strategy

We assign different element types based on physical entity:

- **Walls (2D surfaces)** → `ASDShellQ4` with `ElasticMembranePlateSection`
  - Provides full in-plane and out-of-plane stiffness
  - Required ndf = 6 (3 translations + 3 rotations)

- **Slabs (2D surfaces)** → `ASDShellQ4` with `ElasticMembranePlateSection`
  - Same element type, different section properties
  - Thinner section for floor slabs

- **Columns (1D lines)** → `elasticBeamColumn` with geometric transformation
  - Standard beam-column element for frame action
  - Requires ndf = 6 and geometric transformation
  - Geom transform: "Linear" with vecxz=[1,0,0] defines local x-axis

### Boundary Conditions and Loads

- **Base (z=0)**: Fix all 6 DOFs (fully fixed foundation)
- **Roof level (z=total_h)**: Apply lateral force for pushover analysis


In [ ]:
try:
    print("
Setting up OpenSees model...")
    
    # Initialize OpenSees with 3D, 6 DOF per node
    (g.opensees
        .set_model(ndm=3, ndf=6)
        .build()
    )
    
    print(f"  ✓ OpenSees model initialized (ndm=3, ndf=6)")
    
    # Add section for walls
    ops = g.opensees
    
    try:
        ops.add_section("WallSection", "ElasticMembranePlateSection",
                       E=Ec, nu=nu, h=t_wall, rho=rho_c)
        print(f"  ✓ Added WallSection (E={Ec/1e9:.1f} GPa, h={t_wall:.2f}m)")
    except Exception as e:
        print(f"  Warning adding WallSection: {e}")
    
    # Add section for slabs
    try:
        ops.add_section("SlabSection", "ElasticMembranePlateSection",
                       E=Ec, nu=nu, h=t_slab, rho=rho_c)
        print(f"  ✓ Added SlabSection (E={Ec/1e9:.1f} GPa, h={t_slab:.2f}m)")
    except Exception as e:
        print(f"  Warning adding SlabSection: {e}")
    
    # Add geometric transformation for columns
    try:
        ops.add_geom_transf("LinearTransf", "Linear", vecxz=[1, 0, 0])
        print(f"  ✓ Added LinearTransf for columns")
    except Exception as e:
        print(f"  Warning adding geom_transf: {e}")
    
    # Assign elements to physical groups
    try:
        ops.assign_element("Walls", "ASDShellQ4", section="WallSection")
        print(f"  ✓ Assigned ASDShellQ4 to Walls")
    except Exception as e:
        print(f"  Warning assigning Walls element: {e}")
    
    try:
        ops.assign_element("Slabs", "ASDShellQ4", section="SlabSection")
        print(f"  ✓ Assigned ASDShellQ4 to Slabs")
    except Exception as e:
        print(f"  Warning assigning Slabs element: {e}")
    
    try:
        ops.assign_element("Columns", "elasticBeamColumn",
                          geom_transf="LinearTransf",
                          A=Ac, E=Ec, G=G, Jx=Jx, Iy=Iy, Iz=Iz)
        print(f"  ✓ Assigned elasticBeamColumn to Columns")
    except Exception as e:
        print(f"  Warning assigning Columns element: {e}")
    
    # Apply boundary conditions: fully fixed at base
    try:
        ops.fix("Fixed", dofs=[1, 1, 1, 1, 1, 1])
        print(f"  ✓ Applied fixed constraints at base")
    except Exception as e:
        print(f"  Warning applying constraints: {e}")
    
    # Apply lateral load at roof level
    lateral_force = 100.0e3  # 100 kN lateral force
    try:
        ops.add_nodal_load("Lateral", "RoofLevel",
                          force=[lateral_force, 0.0, 0.0, 0.0, 0.0, 0.0])
        print(f"  ✓ Applied lateral load {lateral_force/1e3:.0f} kN at roof")
    except Exception as e:
        print(f"  Warning applying load: {e}")
    
    print(f"
  OpenSees model setup complete")

except Exception as e:
    print(f"Error during OpenSees setup: {e}")
    import traceback
    traceback.print_exc()
finally:
    pass

In [ ]:
try:
    # Get model summary
    summary = ops.summary()
    print("
── OpenSees Model Summary ──")
    print(summary)
    
    # Node and element tables
    df_nodes = ops.node_table()
    df_elems = ops.element_table()
    
    print(f"
– Nodes and Elements –")
    print(f"Nodes:    {len(df_nodes)}")
    print(f"Elements: {len(df_elems)}")
    
    if len(df_elems) > 0:
        print(f"
Element types:")
        print(df_elems.groupby('ops_type').size())
    
    # Export to Python script
    ops.export_py("building.py")
    print(f"
OpenSees model exported → building.py")

except Exception as e:
    print(f"Error exporting model: {e}")

## Static Lateral Load Analysis

We run a static pushover analysis under the lateral force applied at the roof level. This simulates a seismic-like horizontal load and reveals:

- Building lateral stiffness
- Story drift distribution
- Stress concentration in walls and slabs
- Load path through columns to the foundation

The analysis is nonlinear in geometry (large displacements) but linear in material (elastic).


In [ ]:
import openseespy.opensees as ops

try:
    print("
Running static analysis...")
    
    # Build and analyze using OpenSeesPy
    exec(open("building.py").read())
    
    # Define analysis parameters
    ops.wipeAnalysis()
    ops.constraints('Transformation')
    ops.numberer('RCM')
    ops.system('BandGeneral')
    ops.test('NormUnbalance', 1e-6, 10)
    ops.algorithm('Linear')
    
    # Static analysis
    ops.integrator('LoadControl', 1.0)
    ops.analysis('Static')
    
    # Run analysis
    result = ops.analyze(1)
    
    if result == 0:
        print("  ✓ Analysis completed successfully")
    else:
        print(f"  Warning: Analysis returned code {result}")
    
    # Get nodal displacements at roof level
    roof_nodes = []
    for node_id in range(1, len(df_nodes) + 1):
        try:
            disp = ops.nodeDisp(node_id)
            if len(disp) >= 3:
                node_z = ops.nodeCoord(node_id)[2]
                if abs(node_z - total_h) < 0.1:
                    roof_nodes.append((node_id, disp))
        except:
            pass
    
    print(f"
Roof displacements ({len(roof_nodes)} nodes):")
    if roof_nodes:
        for nid, disp in roof_nodes[:5]:
            print(f"  Node {nid}: ux={disp[0]:.4f}m, uy={disp[1]:.4f}m, uz={disp[2]:.4f}m")

except Exception as e:
    print(f"Error during analysis: {e}")
    import traceback
    traceback.print_exc()

## Post-Processing and Visualization

Extract and visualize key results:

1. **Deformed shape**: Show displaced configuration overlaid with original
2. **Story drift**: Horizontal displacement distribution at each floor
3. **Element forces**: Stress resultants in walls and axial forces in columns


In [ ]:
try:
    # Extract displacements at each story
    story_displacements = []
    
    for story in range(1, n_stories + 1):
        z_target = story * story_h
        nodes_at_story = []
        disp_x_values = []
        
        for node_id in range(1, len(df_nodes) + 1):
            try:
                coord = ops.nodeCoord(node_id)
                node_z = coord[2]
                
                if abs(node_z - z_target) < 0.1:
                    disp = ops.nodeDisp(node_id)
                    nodes_at_story.append(node_id)
                    disp_x_values.append(disp[0])
            except:
                pass
        
        if disp_x_values:
            avg_disp = np.mean(disp_x_values)
            max_disp = np.max(np.abs(disp_x_values))
            story_displacements.append({
                'story': story,
                'z_level': z_target,
                'avg_disp': avg_disp,
                'max_disp': max_disp,
                'drift': avg_disp / story_h
            })
    
    # Plot story displacements
    if story_displacements:
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        
        # Absolute displacement
        stories = [s['story'] for s in story_displacements]
        disps = [s['avg_disp'] for s in story_displacements]
        
        axes[0].bar(stories, disps, color='steelblue', edgecolor='black')
        axes[0].set_xlabel('Story')
        axes[0].set_ylabel('Lateral Displacement [m]')
        axes[0].set_title('Story-Level Displacements (X-direction)')
        axes[0].grid(True, alpha=0.3)
        
        # Story drift ratio
        drifts = [s['drift'] for s in story_displacements]
        axes[1].bar(stories, drifts, color='coral', edgecolor='black')
        axes[1].set_xlabel('Story')
        axes[1].set_ylabel('Drift Ratio')
        axes[1].set_title('Story Drift Ratios')
        axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig('story_results.png', dpi=100, bbox_inches='tight')
        plt.show()
        
        print("Plot saved → story_results.png")
        print("
Story Analysis Results:")
        print("-" * 70)
        for s in story_displacements:
            print(f"Story {s['story']}: disp={s['avg_disp']:.4f}m, "
                  f"drift ratio={s['drift']:.6f}")
    
except Exception as e:
    print(f"Note: Detailed post-processing visualization skipped ({e})")
    print("This is expected in environments without full OpenSees/OpenSeesPy support")


## Key Takeaways

### Multi-Component Assembly
- Combined three structural systems in one model: **walls** (lateral stiffness), **slabs** (floor diaphragms), **columns** (gravity + moment transfer)
- Used **fragment operation** to ensure conformal mesh at all junctions
- All components share nodes automatically after fragmentation

### Mixed Element Types
- **Shell elements** (ASDShellQ4) for 2D surfaces requiring membrane + bending stiffness
- **Beam elements** (elasticBeamColumn) for 1D line elements with full 3D frame action
- Requires **ndf=6** (3 translations + 3 rotations) to couple shells and beams
- Geometric transformation (`LinearTransf`) required for beam elements

### Physical Grouping
- After fragmentation, physical groups are identified by spatial location
- Walls at y ≈ 0 or y ≈ Ly
- Slabs at z = story heights
- Columns as short vertical lines at corners

### Material and Section Modeling
- **Elastic material** for concrete: Young's modulus E = 30 GPa, Poisson's ratio ν = 0.2
- **Shell section**: Thickness property only (elastic membrane-plate behavior)
- **Beam cross-section**: Full geometric properties (A, Jx, Iy, Iz) + elastic modulus

### Analysis Interpretation
- Lateral load induces flexure in walls and shear transfer through slabs
- Story drift indicates building deformation under lateral force
- Base shear distributed among walls and columns according to relative stiffness

### Extensibility
This example can be extended to:
- Add nonlinear material models (concrete damage, steel plasticity)
- Include transverse walls (moment-resisting frames in Y-direction)
- Model openings in walls (doors, windows) via mesh refinement
- Add viscous damping for dynamic analysis
- Perform time-history analysis under earthquake excitation
